# 6. Temporal-Difference Learning

Bu notebook, Sutton & Barto kitabının 6. bölümünü kapsar.

## İçindekiler
1. TD Prediction
2. TD(0) Algoritması
3. SARSA (On-policy TD Control)
4. Q-Learning (Off-policy TD Control)
5. Expected SARSA
6. Maximization Bias ve Double Q-Learning

## 6.1 TD Learning Nedir?

**Temporal-Difference (TD)** learning, MC ve DP'nin en iyi özelliklerini birleştirir:

| Özellik | DP | MC | TD |
|---------|----|----|----|
| Model gerekli | Evet | Hayır | Hayır |
| Bootstrapping | Evet | Hayır | Evet |
| Episode bitmeli | Hayır | Evet | Hayır |

### TD'nin Temel Fikri

MC: Episode sonunda tam return ile güncelle:
$$V(S_t) \leftarrow V(S_t) + \alpha [G_t - V(S_t)]$$

TD: Her adımda **tahmin** ile güncelle (bootstrapping):
$$V(S_t) \leftarrow V(S_t) + \alpha [R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

### TD Error

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

Bu **TD error**, öğrenme sinyalidir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Tuple, List

class CliffWalkingEnv:
    """
    Cliff Walking Environment (Example 6.6 from Sutton & Barto).
    
    4x12 grid. Start: bottom-left. Goal: bottom-right.
    Cliff: bottom row (except start and goal).
    Falling off cliff: -100 reward, back to start.
    Other moves: -1 reward.
    """
    
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)  # Bottom-left
        self.goal = (3, 11)  # Bottom-right
        self.cliff = [(3, i) for i in range(1, 11)]  # Bottom row except corners
        
        self.n_states = self.rows * self.cols
        self.n_actions = 4  # up, right, down, left
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        
        self.reset()
    
    def reset(self):
        self.position = self.start
        return self.position
    
    def step(self, action):
        row, col = self.position
        drow, dcol = self.actions[action]
        
        new_row = max(0, min(self.rows - 1, row + drow))
        new_col = max(0, min(self.cols - 1, col + dcol))
        new_pos = (new_row, new_col)
        
        # Check if fell off cliff
        if new_pos in self.cliff:
            self.position = self.start
            return self.position, -100, False
        
        self.position = new_pos
        
        # Check if reached goal
        if self.position == self.goal:
            return self.position, -1, True
        
        return self.position, -1, False

env = CliffWalkingEnv()
print(f"Start: {env.start}, Goal: {env.goal}")
print(f"Cliff positions: {env.cliff}")

In [ ]:
def visualize_cliff_world(env, V=None, Q=None, policy=None, path=None, title="Cliff Walking"):
    """Cliff walking environment'ı görselleştir."""
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Draw grid
    for row in range(env.rows):
        for col in range(env.cols):
            pos = (row, col)
            
            if pos == env.start:
                color = 'lightgreen'
                label = 'S'
            elif pos == env.goal:
                color = 'gold'
                label = 'G'
            elif pos in env.cliff:
                color = 'gray'
                label = 'C'
            else:
                color = 'white'
                label = ''
            
            rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                                 facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            
            if label:
                ax.text(col + 0.5, env.rows - row - 0.5, label,
                       ha='center', va='center', fontsize=12, fontweight='bold')
    
    # Draw path if provided
    if path:
        path_x = [p[1] + 0.5 for p in path]
        path_y = [env.rows - p[0] - 0.5 for p in path]
        ax.plot(path_x, path_y, 'b-o', linewidth=2, markersize=4, alpha=0.7)
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.show()

visualize_cliff_world(env)

## 6.2 TD(0) Prediction

Verilen policy için V(s) tahmin et.

### Algoritma

```
Initialize V(s) arbitrarily
For each episode:
    Initialize S
    For each step:
        A ← action from π for S
        Take action A, observe R, S'
        V(S) ← V(S) + α[R + γV(S') - V(S)]
        S ← S'
    Until S is terminal
```

In [ ]:
def td_prediction(env, policy, n_episodes=500, alpha=0.1, gamma=1.0):
    """
    TD(0) Prediction.
    
    Args:
        env: Environment
        policy: Function state -> action
        n_episodes: Number of episodes
        alpha: Learning rate
        gamma: Discount factor
    
    Returns:
        V: State value function
    """
    V = defaultdict(float)
    
    for episode in range(n_episodes):
        state = env.reset()
        
        while True:
            action = policy(state)
            next_state, reward, done = env.step(action)
            
            # TD(0) update
            td_target = reward + gamma * V[next_state]
            td_error = td_target - V[state]
            V[state] += alpha * td_error
            
            if done:
                break
            
            state = next_state
    
    return V

# Test with a simple policy (always go right, if not possible go down)
def simple_policy(state):
    row, col = state
    if col < 11:
        return 1  # right
    return 2  # down

V = td_prediction(env, simple_policy, n_episodes=1000)
print(f"Estimated values for {len(V)} states")

## 6.3 SARSA: On-policy TD Control

**SARSA** = State-Action-Reward-State-Action

### Update Rule

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

### Özellikler
- **On-policy**: Aynı ε-greedy policy'yi hem öğrenmek hem de explore etmek için kullanır
- Behavior policy = Target policy

In [ ]:
def sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    SARSA (On-policy TD Control).
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(state)
        
        total_reward = 0
        
        while True:
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            next_action = epsilon_greedy(next_state)
            
            # SARSA update
            td_target = reward + gamma * Q[next_state][next_action]
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
            action = next_action
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards

Q_sarsa, rewards_sarsa = sarsa(env, n_episodes=500)
print(f"SARSA trained for {len(Q_sarsa)} states")

## 6.4 Q-Learning: Off-policy TD Control

### Update Rule

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)]$$

### Özellikler
- **Off-policy**: Greedy policy'yi (target) öğrenirken ε-greedy (behavior) ile explore eder
- Target policy: $\pi(s) = \arg\max_a Q(s, a)$ (greedy)
- Behavior policy: ε-greedy
- "Optimistic" learning - en iyi olası sonucu varsayar

In [ ]:
def q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    Q-Learning (Off-policy TD Control).
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        while True:
            action = epsilon_greedy(state)
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # Q-Learning update (max over next actions)
            td_target = reward + gamma * np.max(Q[next_state])
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards

Q_qlearning, rewards_qlearning = q_learning(env, n_episodes=500)
print(f"Q-Learning trained for {len(Q_qlearning)} states")

In [ ]:
# SARSA vs Q-Learning karşılaştırması
def compare_algorithms(env, n_runs=10, n_episodes=500):
    """SARSA ve Q-Learning'i karşılaştır."""
    
    sarsa_rewards = np.zeros((n_runs, n_episodes))
    qlearn_rewards = np.zeros((n_runs, n_episodes))
    
    for run in range(n_runs):
        _, rewards_s = sarsa(env, n_episodes=n_episodes)
        _, rewards_q = q_learning(env, n_episodes=n_episodes)
        
        sarsa_rewards[run] = rewards_s
        qlearn_rewards[run] = rewards_q
    
    return sarsa_rewards.mean(axis=0), qlearn_rewards.mean(axis=0)

sarsa_avg, qlearn_avg = compare_algorithms(env, n_runs=10, n_episodes=500)

# Plot
plt.figure(figsize=(12, 5))

# Smooth with moving average
window = 10
sarsa_smooth = np.convolve(sarsa_avg, np.ones(window)/window, mode='valid')
qlearn_smooth = np.convolve(qlearn_avg, np.ones(window)/window, mode='valid')

plt.plot(sarsa_smooth, label='SARSA', color='blue')
plt.plot(qlearn_smooth, label='Q-Learning', color='red')
plt.xlabel('Episode')
plt.ylabel('Sum of Rewards')
plt.title('SARSA vs Q-Learning on Cliff Walking')
plt.legend()
plt.ylim(-100, 0)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def extract_path(env, Q):
    """Q'dan greedy policy ile path çıkar."""
    path = []
    state = env.reset()
    path.append(state)
    
    for _ in range(100):  # Max steps
        action = np.argmax(Q[state])
        next_state, _, done = env.step(action)
        path.append(next_state)
        
        if done:
            break
        state = next_state
    
    return path

# Visualize learned paths
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# SARSA path
ax = axes[0]
path_sarsa = extract_path(env, Q_sarsa)
for row in range(env.rows):
    for col in range(env.cols):
        pos = (row, col)
        if pos == env.start:
            color = 'lightgreen'
        elif pos == env.goal:
            color = 'gold'
        elif pos in env.cliff:
            color = 'gray'
        else:
            color = 'white'
        rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                             facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)

path_x = [p[1] + 0.5 for p in path_sarsa]
path_y = [env.rows - p[0] - 0.5 for p in path_sarsa]
ax.plot(path_x, path_y, 'b-o', linewidth=2, markersize=6)
ax.set_xlim(0, env.cols)
ax.set_ylim(0, env.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('SARSA: Safe Path', fontsize=14)

# Q-Learning path
ax = axes[1]
path_qlearn = extract_path(env, Q_qlearning)
for row in range(env.rows):
    for col in range(env.cols):
        pos = (row, col)
        if pos == env.start:
            color = 'lightgreen'
        elif pos == env.goal:
            color = 'gold'
        elif pos in env.cliff:
            color = 'gray'
        else:
            color = 'white'
        rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                             facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)

path_x = [p[1] + 0.5 for p in path_qlearn]
path_y = [env.rows - p[0] - 0.5 for p in path_qlearn]
ax.plot(path_x, path_y, 'r-o', linewidth=2, markersize=6)
ax.set_xlim(0, env.cols)
ax.set_ylim(0, env.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Q-Learning: Optimal Path', fontsize=14)

plt.tight_layout()
plt.show()

## 6.5 Expected SARSA

Q-Learning ve SARSA'nın ortası:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma \sum_a \pi(a|S_{t+1}) Q(S_{t+1}, a) - Q(S_t, A_t)]$$

### Avantajları
- SARSA'dan daha düşük variance
- Hem on-policy hem off-policy olabilir

In [ ]:
def expected_sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    Expected SARSA.
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    def expected_value(state):
        """E[Q(s,a)] under ε-greedy policy."""
        q_values = Q[state]
        best_action = np.argmax(q_values)
        
        # ε-greedy action probabilities
        policy_probs = np.ones(env.n_actions) * epsilon / env.n_actions
        policy_probs[best_action] += 1 - epsilon
        
        return np.dot(policy_probs, q_values)
    
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        while True:
            action = epsilon_greedy(state)
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # Expected SARSA update
            td_target = reward + gamma * expected_value(next_state)
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards

Q_expected, rewards_expected = expected_sarsa(env, n_episodes=500)
print(f"Expected SARSA trained")

## 6.6 Maximization Bias ve Double Q-Learning

### Problem: Maximization Bias

Q-Learning'de $\max_a Q(s', a)$ kullanımı **pozitif bias** yaratır.

Neden? $E[\max(X_1, X_2, ...)] \geq \max(E[X_1], E[X_2], ...)$

### Çözüm: Double Q-Learning

İki Q fonksiyonu kullan:
- $Q_1$: Action seçimi için
- $Q_2$: Value tahmini için (veya tersi)

$$Q_1(S, A) \leftarrow Q_1(S, A) + \alpha [R + \gamma Q_2(S', \arg\max_a Q_1(S', a)) - Q_1(S, A)]$$

In [ ]:
def double_q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    Double Q-Learning.
    """
    Q1 = defaultdict(lambda: np.zeros(env.n_actions))
    Q2 = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        # Use sum of Q1 and Q2 for action selection
        return np.argmax(Q1[state] + Q2[state])
    
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        while True:
            action = epsilon_greedy(state)
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # Randomly update Q1 or Q2
            if np.random.random() < 0.5:
                # Update Q1
                best_action = np.argmax(Q1[next_state])
                td_target = reward + gamma * Q2[next_state][best_action]
                Q1[state][action] += alpha * (td_target - Q1[state][action])
            else:
                # Update Q2
                best_action = np.argmax(Q2[next_state])
                td_target = reward + gamma * Q1[next_state][best_action]
                Q2[state][action] += alpha * (td_target - Q2[state][action])
            
            if done:
                break
            
            state = next_state
        
        episode_rewards.append(total_reward)
    
    # Combine Q1 and Q2
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    all_states = set(Q1.keys()) | set(Q2.keys())
    for s in all_states:
        Q[s] = (Q1[s] + Q2[s]) / 2
    
    return Q, episode_rewards

Q_double, rewards_double = double_q_learning(env, n_episodes=500)
print("Double Q-Learning trained")

In [ ]:
# Tüm algoritmaları karşılaştır
n_runs = 10
n_episodes = 500

algorithms = {
    'SARSA': sarsa,
    'Q-Learning': q_learning,
    'Expected SARSA': expected_sarsa,
    'Double Q-Learning': double_q_learning
}

results = {}
for name, algo in algorithms.items():
    all_rewards = []
    for _ in range(n_runs):
        _, rewards = algo(env, n_episodes=n_episodes)
        all_rewards.append(rewards)
    results[name] = np.mean(all_rewards, axis=0)

# Plot
plt.figure(figsize=(12, 6))
colors = ['blue', 'red', 'green', 'purple']
window = 10

for (name, rewards), color in zip(results.items(), colors):
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=name, color=color, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Sum of Rewards')
plt.title('TD Control Algorithms Comparison')
plt.legend()
plt.ylim(-100, 0)
plt.grid(True, alpha=0.3)
plt.show()

## Özet

| Algoritma | Tip | Update Target |
|-----------|-----|---------------|
| **TD(0)** | Prediction | $R + \gamma V(S')$ |
| **SARSA** | On-policy Control | $R + \gamma Q(S', A')$ |
| **Q-Learning** | Off-policy Control | $R + \gamma \max_a Q(S', a)$ |
| **Expected SARSA** | Both | $R + \gamma E_\pi[Q(S', a)]$ |
| **Double Q** | Off-policy Control | Decoupled selection & evaluation |

### SARSA vs Q-Learning

| | SARSA | Q-Learning |
|--|-------|------------|
| Policy | On-policy | Off-policy |
| Öğrendiği | ε-greedy policy'nin değeri | Optimal policy'nin değeri |
| Davranış | Daha "güvenli" | Daha "riskli" |
| Cliff örneğinde | Uçurumdan uzak | Uçurum kenarından |

### Sonraki Notebook
**07 - N-Step Bootstrapping**: TD ve MC arasındaki spektrum